# 25 · Fine-tuning and LoRA

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** optional. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Understand trainable adapters; optionally fine-tune a tiny randomly initialized causal transformer on CPU.


## Step 1 — Set the scope

This self-contained lab builds a tiny GPT-2 architecture from configuration, so no pretrained weights are downloaded. It teaches causal-LM loss and LoRA mechanics; it is not useful language-model adaptation. Add `torch>=2.2,<3`, `transformers>=4.45,<5`, and `peft>=0.13,<1` to the environment first. Real pretrained tuning is an extension.


In [ ]:
RUN_TRAINING=False
print('Tiny CPU LoRA training enabled:',RUN_TRAINING)


**Check your result:** Training is off until the optional packages are installed and you opt in.


## Step 2 — Create a tiny transformer and adapters

LoRA freezes the base parameters and trains low-rank matrices. GPT-2 uses Conv1D projections, which require fan_in_fan_out=True.


In [ ]:
if RUN_TRAINING:
    import torch
    from transformers import GPT2Config, GPT2LMHeadModel
    from peft import LoraConfig, get_peft_model, TaskType
    torch.manual_seed(42)
    config=GPT2Config(vocab_size=32,n_positions=32,n_ctx=32,n_embd=32,n_layer=1,n_head=2,bos_token_id=1,eos_token_id=2,pad_token_id=0)
    base=GPT2LMHeadModel(config)
    model=get_peft_model(base,LoraConfig(task_type=TaskType.CAUSAL_LM,r=4,lora_alpha=8,
        lora_dropout=0.0,target_modules=['c_attn'],fan_in_fan_out=True))
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in model.parameters())
    assert 0<trainable<total
    print({'trainable':trainable,'total':total})
else:
    print('Adapter creation skipped')


**Check your result:** Enabled mode reports fewer trainable parameters than total parameters.


## Step 3 — Train using causal LM loss

Tokens are synthetic integers. The model shifts labels internally so position t predicts token t+1. All sequences are full length, so no padding loss mask is needed here.


In [ ]:
if RUN_TRAINING:
    tokens=torch.tensor([[1,3,4,5,6,7,8,2]]*8,dtype=torch.long)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=0.01)
    model.train()
    losses=[]
    for step in range(30):
        optimizer.zero_grad()
        loss=model(input_ids=tokens,attention_mask=torch.ones_like(tokens),labels=tokens).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        losses.append(float(loss.detach()))
    assert all(__import__('math').isfinite(v) for v in losses)
    print('First/last loss:',losses[0],losses[-1])
else:
    print('Training skipped; no GPU, downloads or endpoint needed for the default run.')


**Check your result:** Enabled mode completes 30 bounded optimization steps.


## Step 4 — Save the adapter optionally

Adapter files must be saved with the matching base configuration/checkpoint for reproducible reuse. Use a writable Unity Catalog volume path; never assume /tmp is durable.


In [ ]:
ADAPTER_PATH='' # Example: /Volumes/<catalog>/<schema>/<volume>/tiny_lora
if RUN_TRAINING and ADAPTER_PATH:
    assert ADAPTER_PATH.startswith('/Volumes/')
    model.save_pretrained(ADAPTER_PATH)
    base.config.save_pretrained(ADAPTER_PATH)
    # This randomly initialized base must also be preserved for adapter reuse.
    import os
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},os.path.join(ADAPTER_PATH,'full_training_state.pt'))
    print('Adapter, config and complete matching model state saved.')
else:
    print('No artifacts saved. Set a volume path to preserve this optional experiment.')


**Check your result:** No files are written by default.


## Practice

Adapt a permitted small pretrained model, keeping a held-out validation set. Mask padding labels with −100. Compare the base model and adapter on task metrics.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Pin the base model revision, tokenizer, prompt format, dataset license and package versions. Random-base training demonstrates optimization only; do not interpret its loss as language quality.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
